# Giai đoạn 1 — huấn luyện trên Google Colab
Chạy notebook này trên **CPU** khi cần huấn luyện đầy đủ. Kết quả ghi ngay vào Drive; cuối cùng tải **san_pham_giai_doan_1.zip** (mô hình best + JSON, kết quả kiểm tra, bảng/hình Chương 5, so sánh cùng J, GCD-Sim và manifest SHA256). Giai đoạn 2 nhận ZIP trên máy local để hoàn thiện ứng dụng terminal và chép tư liệu báo cáo; không huấn luyện lại các thực nghiệm.

Nguồn mục tiêu/số liệu: “Dữ liệu điện hóa.docx” — NTM.Hồng & LD.Phúc. Dự đoán Csp/V̄, suy Ed/Pd; so sánh 8 điện cực **cùng J**; KB4 chỉ phân tích giới hạn với điện cực mới. Không áp đặt ratio 0,1 là tốt nhất. Tài liệu Tân/Giang/cô Nhạn chỉ tham khảo.

Mỗi phiên mới: chạy lại **Thiết lập 1–4**, rồi nhảy đến Phiên đang dở. Chạy lại lệnh = tiếp tục, bỏ đơn vị đủ seed. Không mở hai bộ chạy ghi cùng THU_MUC. Chỉ điểm thật bị giấu mới là kết quả kiểm tra; mô hình ứng dụng là **khớp lại để sử dụng, không phải kết quả kiểm tra**.

Xem bảng thời gian và phương pháp ngoại suy local → Colab trong README; chưa đo thời gian trên Colab. GCD-Sim 480.000 mẫu có thể mất khoảng 1–3 phút trên Colab (ước lượng, chưa đo); XLSX khoảng 53,5 MB, **~60 giây local cho cả lệnh (đo sau khi chuyển sang ghi tuần tự)**, không lưu trong data/ hoặc Git.

## Thiết lập 1 — kết nối Drive
Chạy đầu **mỗi Phiên 1/2/3**, kể cả sau mất kết nối. Kết quả lâu dài nằm ở MyDrive/ScientificProject/ket_qua; /content có thể mất khi Colab ngắt.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Thiết lập 2 — lấy đúng mã nguồn
Chạy sau mount Drive trong mỗi phiên. **Đẩy các thay đổi local lên GitHub trước** (tự commit/push ngoài notebook); clone/pull chỉ nhận mã đã có trên remote. REPO_URL có sẵn URL chuẩn. Không muốn đẩy: đặt DUNG_ZIP=True, tải ZIP **mã nguồn hiện tại** có ml/, gcdsim/, data/, requirements.txt. Mã làm việc ở /content/Code; khi clone, bản sao tại đây không có .git sẽ được xóa trước. Chỉ lưu kết quả trên Drive; không dùng ZIP sản phẩm ở bước lấy mã.

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

REPO_URL = 'https://github.com/HMHPRSA2005/graphene-supercap-ml'
DUNG_ZIP = False
THU_MUC_MA = Path('/content/Code')


def chay_git(*lenh):
    """Chạy git và in đủ thông báo; subprocess không in stderr của git ra ô Colab."""
    kq = subprocess.run(['git', *lenh], capture_output=True, text=True)
    print(kq.stdout + kq.stderr)
    if kq.returncode != 0:
        raise RuntimeError(
            f'git {lenh[0]} thất bại (mã {kq.returncode}); đọc dòng "fatal:" ngay phía trên. '
            'Thường gặp: REPO_URL sai hoặc repo để private (dùng DUNG_ZIP = True), '
            'mạng tạm lỗi (chạy lại ô), /content/Code còn sót (Thời gian chạy > Ngắt kết nối và xóa thời gian chạy).')


os.chdir('/content')
if DUNG_ZIP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chỉ tải một ZIP mã nguồn.')
    noi_giai_nen = Path('/content/ma_tai_len')
    noi_giai_nen.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for muc in z.infolist():
            if not (noi_giai_nen / muc.filename).resolve().is_relative_to(noi_giai_nen.resolve()):
                raise ValueError('ZIP có đường dẫn vượt thư mục nhận.')
        z.extractall(noi_giai_nen)
    cac_goc = [p.parent for p in noi_giai_nen.rglob('requirements.txt')
               if all((p.parent / thu).is_dir() for thu in ('ml', 'gcdsim', 'data'))]
    if len(cac_goc) != 1:
        raise ValueError('Cần đúng một gốc mã nguồn có ml/, gcdsim/, data/, requirements.txt.')
    shutil.copytree(cac_goc[0], THU_MUC_MA, dirs_exist_ok=True)
elif (THU_MUC_MA / '.git').is_dir():
    chay_git('-C', str(THU_MUC_MA), 'pull', '--ff-only')
else:
    if THU_MUC_MA.exists():
        print('Xóa bản sao mã /content/Code không có .git trước khi clone; kết quả nằm trên Drive.')
        shutil.rmtree(THU_MUC_MA)
    chay_git('clone', REPO_URL, str(THU_MUC_MA))
os.chdir(THU_MUC_MA)
print('Mã nguồn:', Path.cwd())


## Thiết lập 3 — cài thư viện và ghi nhận CPU
Chạy sau lấy mã trong mỗi phiên. Kết quả phiên bản/CPU hiện dưới ô; lịch sử thực nghiệm còn được ghi vào THU_MUC/moi_truong.json. Không cần GPU. Nếu pip yêu cầu khởi động lại runtime, làm theo rồi chạy lại Thiết lập 1–4.

In [ ]:
%pip install -r requirements.txt
import importlib.metadata
import platform
print('Python:', platform.python_version())
print('CPU:', platform.processor(), '| Số nhân:', os.cpu_count())
!lscpu
for ten in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib', 'matplotlib', 'openpyxl']:
    print(ten, importlib.metadata.version(ten))


## Thiết lập 4 — thư mục kết quả và tự kiểm ghi
Chạy sau cài thư viện trong mỗi phiên, **trước mọi thực nghiệm**. Kết quả/resume nằm đúng THU_MUC dưới đây. Nếu lỗi ghi/phục hồi, sửa kết nối Drive trước khi tiếp tục; không đổi sang thư mục tạm để chạy đầy đủ.

In [ ]:
THU_MUC = '/content/drive/MyDrive/ScientificProject/ket_qua'
Path(THU_MUC).mkdir(parents=True, exist_ok=True)
print('Kết quả / resume:', THU_MUC)
!python -X utf8 -m ml.chay_kb1_kb4 --kiem-tra-ghi --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## GCD-Sim — N=10000 mỗi điện cực
Chạy sau thiết lập, trước hoặc sau Phiên 1. XLSX chỉ được bỏ qua sau kiểm `is_zipfile` và `testzip()` không lỗi CRC; tệp hỏng/ghi dở được sinh lại. Kết quả ở THU_MUC/gcdsim/: 48 × 10000 = **480.000 mẫu ảo** trong du_lieu_ao_480000.xlsx, hình và do_nhay.csv. Ước lượng Colab 1–3 phút cho XLSX, **chưa đo trên Colab**; **~60 giây local cho cả lệnh (đo sau khi chuyển sang ghi tuần tự)**, XLSX ~53,5 MB. XLSX lớn không đưa lên Git nhưng có trong ZIP. Chỉ để trình bày GCD-Sim, **không dùng làm bảng học chung trước chia fold**.

GCD-Sim ghi XLSX qua tệp tạm rồi đổi tên nguyên tử; nếu lệnh bị ngắt, chạy lại ô này, XLSX thiếu/hỏng sẽ không được bỏ qua. Hình/CSV vẫn bỏ qua nếu có đầu ra; xóa đúng đầu ra hình/CSV đang ghi dở nếu cần sinh lại. Chi tiết xem gcdsim/README.md.

In [ ]:
from ml.dong_goi import xlsx_nguyen_ven
if not xlsx_nguyen_ven(Path(THU_MUC) / 'gcdsim/du_lieu_ao_480000.xlsx'):
    !python -X utf8 -m gcdsim.gcdsim data/so_lieu_GCD.xlsx --out "$THU_MUC/gcdsim/du_lieu_ao_480000.xlsx" --seed 42
    if _exit_code != 0:
        raise RuntimeError('GCD-Sim thất bại; sửa lỗi rồi chạy lại.')
else:
    print('Đã có XLSX GCD-Sim trên Drive, bỏ qua.')


In [ ]:
if not any((Path(THU_MUC) / 'gcdsim/hinh').glob('*.png')):
    !python -X utf8 -m gcdsim.ve_hinh data/so_lieu_GCD.xlsx "$THU_MUC/gcdsim/hinh"
    if _exit_code != 0:
        raise RuntimeError('Vẽ GCD-Sim thất bại.')
else:
    print('Đã có hình GCD-Sim, bỏ qua.')


In [ ]:
if not (Path(THU_MUC) / 'gcdsim/do_nhay.csv').is_file() or (Path(THU_MUC) / 'gcdsim/do_nhay.csv').stat().st_size == 0:
    !python -X utf8 -m gcdsim.do_nhay data/so_lieu_GCD.xlsx "$THU_MUC/gcdsim/do_nhay.csv"
    if _exit_code != 0:
        raise RuntimeError('Độ nhạy GCD-Sim thất bại.')
else:
    print('Đã có do_nhay.csv, bỏ qua.')


## Chạy thử môi trường — tùy chọn, không phải thực nghiệm đầy đủ
Chạy một lần sau thiết lập nếu muốn bắt lỗi sớm. Chỉ 2 đơn vị nhóm thu; kết quả ở **/content/chay_thu**, không trộn vào Drive và không đưa vào sản phẩm. Đây là chạy thử local của runtime, không phải một giai đoạn.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom thu --thu-muc /content/chay_thu --toi-da-don-vi 2 --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Kế hoạch Phiên 1/2/3
Chạy theo thứ tự; kết quả mọi phiên ở THU_MUC. Mỗi phiên mới chạy lại Thiết lập 1–4. Nếu hạn mức hết trước ước lượng, chạy lại đúng lệnh ở phiên sau; không xóa CSV. Các nhóm có đơn vị trùng được bỏ qua bằng cùng khóa và seed, không dựa vào cột nhom. Thời gian ngoại suy local → Colab 2 nhân, **chưa đo trên Colab**; xem bảng số đo/phương pháp trong README.

| Phiên | Nhóm | Ước lượng Colab 2 nhân | Sản phẩm cuối phiên |
|---|---|---|---|
| 1 | chinh, du_lieu_ao | ~5,5 giờ | kết quả kiểm tra + best bản đầu |
| 2 | so_mau (10/100/1000/10000), sigma | ~7,1 giờ | cập nhật kết quả + best |
| 3 | kb3, kb4, so_sanh, ve_hinh_ch5, đóng gói | ~2,7 giờ + xuất sản phẩm | bảng/hình + ZIP sản phẩm |

Đo local một đơn vị đủ 10 seed tại ×10000, gồm chọn tham số/sinh/học/dự đoán: KB1 MLP/SVR/GPR = 159,60/11,88/44,29 giây; KB2 = 146,41/10,95/34,02 giây (một luồng, 470.047 hàng/fold trước lấy mẫu con). Giả định worker Colab chậm 2× local, chạy 2 worker: tổng local ×2/2. Mức mới thêm 120 đơn vị ≈2,34 giờ; so_mau từ ~1,9 lên ~4,3 giờ. Các nhóm khác giữ ước lượng cũ: chinh 1,8; du_lieu_ao 3,7; sigma 2,8; kb3 2,2; kb4 0,5 giờ, chưa đo lại toàn nhóm. Tổng ~15,3 giờ/2104 đơn vị mới, chưa đo trên Colab. Phiên 2 chưa vượt ~10 giờ nên giữ chung so_mau+sigma; nếu CPU/hạn mức thực tế chậm hơn thì ngắt và resume ở phiên sau.

Best được khớp lại toàn bộ 48 điểm **để sử dụng, không phải kết quả kiểm tra**. Best chung tối thiểu hóa trung bình MAPE(Csp, Pd) của KB1/KB2; best_co_khoang chỉ xét mô hình có sd. Chỉ cấu hình đủ lần chia mới được chọn; không dùng --cho-phep-thieu trong luồng chính.

## Phiên 1 — huấn luyện
Chạy sau Thiết lập 1–4 và sau phiên trước; nếu bị ngắt thì chạy lại chính các ô này. CSV ghi ngay tại THU_MUC/ket_qua_kb1_kb4.csv; mỗi nhóm giữ đủ 10 seed, lưới đầy đủ. Không chạy hai ô thực nghiệm đồng thời.

### Nhóm chinh
Chạy khi đến Phiên 1, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom chinh --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Nhóm du_lieu_ao
Chạy khi đến Phiên 1, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom du_lieu_ao --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Lưu best sau Phiên 1
Chạy khi các nhóm của phiên xong. Kết quả ở THU_MUC/best/: best_KB1, best_KB2, best_chung, best_co_khoang, mỗi mục gồm .joblib + .json và bảng xếp hạng. Chạy lại sẽ cập nhật best theo kết quả hiện có.

In [ ]:
!python -X utf8 -m ml.luu_best --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Phiên 2 — huấn luyện
Chạy sau Thiết lập 1–4 và sau phiên trước; nếu bị ngắt thì chạy lại chính các ô này. CSV ghi ngay tại THU_MUC/ket_qua_kb1_kb4.csv; mỗi nhóm giữ đủ 10 seed, lưới đầy đủ. Không chạy hai ô thực nghiệm đồng thời.

### Nhóm so_mau
Chạy khi đến Phiên 2, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom so_mau --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Nhóm sigma
Chạy khi đến Phiên 2, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom sigma --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Lưu best sau Phiên 2
Chạy khi các nhóm của phiên xong. Kết quả ở THU_MUC/best/: best_KB1, best_KB2, best_chung, best_co_khoang, mỗi mục gồm .joblib + .json và bảng xếp hạng. Chạy lại sẽ cập nhật best theo kết quả hiện có.

In [ ]:
!python -X utf8 -m ml.luu_best --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Phiên 3 — huấn luyện
Chạy sau Thiết lập 1–4 và sau phiên trước; nếu bị ngắt thì chạy lại chính các ô này. CSV ghi ngay tại THU_MUC/ket_qua_kb1_kb4.csv; mỗi nhóm giữ đủ 10 seed, lưới đầy đủ. Không chạy hai ô thực nghiệm đồng thời.

### Nhóm kb3
Chạy khi đến Phiên 3, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom kb3 --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Nhóm kb4
Chạy khi đến Phiên 3, hoặc khi tiến độ báo nhóm này còn thiếu. Đầu ra nối tiếp trong THU_MUC; chạy lại = resume.

In [ ]:
!python -X utf8 -m ml.chay_kb1_kb4 --nhom kb4 --thu-muc "$THU_MUC" --n-jobs -1
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


### Lưu best sau Phiên 3
Chạy khi các nhóm của phiên xong. Kết quả ở THU_MUC/best/: best_KB1, best_KB2, best_chung, best_co_khoang, mỗi mục gồm .joblib + .json và bảng xếp hạng. Chạy lại sẽ cập nhật best theo kết quả hiện có.

In [ ]:
!python -X utf8 -m ml.luu_best --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Xem tiến độ bất kỳ lúc nào giữa hai lệnh huấn luyện
Chạy sau thiết lập hoặc sau một nhóm; không chạy khi bộ chạy đang ghi CSV. Chỉ in số đơn vị đủ mọi seed so với yêu cầu và các mục còn thiếu; chưa tạo ZIP. Kết quả in ngay dưới ô, nhóm thu không bắt buộc.

In [ ]:
!python -X utf8 -m ml.dong_goi --thu-muc "$THU_MUC" --chi-tien-do
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Cuối Phiên 3 — so sánh và sinh tư liệu Chương 5
Chạy **sau khi sáu nhóm chính đầy đủ**. So sánh 8 điện cực cùng J ở THU_MUC/so_sanh/; bảng CSV/TeX ở bang_ch5/, hình PDF/PNG ở hinh_ch5/. --ung-dung vẽ thêm Hình 5.9–5.10 bằng GPR cách A khớp toàn bộ, không phải kết quả kiểm tra. Không suy rằng tăng cường luôn tốt hơn A; chọn theo Bảng 5.3. Khoảng 95% là biên, không phải khoảng đồng thời; xác suất tốt nhất giả định các điện cực độc lập.

In [ ]:
!python -X utf8 -m ml.so_sanh --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")
!python -X utf8 -m ml.ve_hinh_ch5 --thu-muc "$THU_MUC" --ung-dung
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")
!python -X utf8 -m ml.luu_best --thu-muc "$THU_MUC"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")


## Đóng gói và tải sản phẩm Giai đoạn 1
Chạy khi tiến độ đủ và GCD-Sim/best/bảng/hình đã có. ZIP lưu ở **/content/drive/MyDrive/ScientificProject/san_pham_giai_doan_1.zip**; files.download tải bản đó về máy. Bản gốc THU_MUC vẫn trên Drive để resume. Không thêm --cho-phep-thieu: nếu thiếu, chạy đúng các lệnh được thông báo rồi đóng gói lại.

Giai đoạn 2 trên máy local: đặt ZIP trong Code/, cài requirements, dùng thư mục ket_qua **mới/rỗng**, chạy `python -X utf8 -m ml.giai_doan_2 tat_ca san_pham_giai_doan_1.zip --thu-muc ket_qua --bao-cao ../Report/Report`. Nếu trùng tệp, lệnh mặc định từ chối; chỉ thêm --ghi-de sau sao lưu. Chỉ mở joblib do chính bạn tạo/tin cậy; SHA256 không xác minh tác giả. Xem README để hoàn thiện báo cáo; nếu kiểm kê báo sản phẩm cũ hơn CSV, chạy lại ô xuất tư liệu/best cuối rồi đóng gói.

In [ ]:
SAN_PHAM = '/content/drive/MyDrive/ScientificProject/san_pham_giai_doan_1.zip'
!python -X utf8 -m ml.dong_goi --thu-muc "$THU_MUC" --out "$SAN_PHAM"
if _exit_code != 0:
    raise RuntimeError("Lệnh thất bại; đọc lỗi phía trên, sửa rồi chạy lại ô này.")
from google.colab import files
files.download(SAN_PHAM)


## Phụ lục — mặc định không chạy
Chỉ chạy các ô sau khi thực sự cần phụ lục, sau thiết lập. Đổi CHAY_PHU_LUC=True; kết quả riêng ở **THU_MUC + '/phu_luc'**, bảng/hình có tiền tố phu_luc_, không trộn với best chính. PGCN-R tự cài theo GNG/LLCS vì chưa có mã gốc, không khẳng định tái lập khóa luận Tân. Sau khi chạy phụ lục, sinh lại tư liệu trong thư mục phụ lục và đóng gói lại nếu muốn kèm sản phẩm.

In [ ]:
CHAY_PHU_LUC = False
CHAY_PHAN_TICH_THU_TU = False  # Đảo thứ tự KB5 làm chi phí xấp xỉ 6 lần.
THU_MUC_PHU_LUC = THU_MUC + '/phu_luc'
if CHAY_PHU_LUC:
    !python -X utf8 -m ml.chay_kb1_kb4 --nhom phu_luc_do_thi --thu-muc "$THU_MUC_PHU_LUC" --n-jobs -1
    if _exit_code != 0:
        raise RuntimeError('Phụ lục đồ thị thất bại.')


### Phụ lục KB5
Chạy chỉ khi bật CHAY_PHU_LUC ở trên. Kết quả nối tiếp trong THU_MUC/phu_luc/ket_qua_kb5.csv; đảo thứ tự còn yêu cầu CHAY_PHAN_TICH_THU_TU=True. Sinh lại bảng/hình trong THU_MUC/phu_luc khi xong; không ghi vào CSV huấn luyện chính.

In [ ]:
if CHAY_PHU_LUC:
    !python -X utf8 -m ml.chay_kb5 --thu-muc "$THU_MUC_PHU_LUC"
    if _exit_code != 0:
        raise RuntimeError('KB5 thất bại.')
if CHAY_PHU_LUC and CHAY_PHAN_TICH_THU_TU:
    !python -X utf8 -m ml.chay_kb5 --thu-muc "$THU_MUC_PHU_LUC" --doi-thu-tu
    if _exit_code != 0:
        raise RuntimeError('Đảo thứ tự KB5 thất bại.')
if CHAY_PHU_LUC:
    !python -X utf8 -m ml.ve_hinh_ch5 --thu-muc "$THU_MUC_PHU_LUC" --ung-dung
    if _exit_code != 0:
        raise RuntimeError('Sinh hình phụ lục thất bại.')
